# Fine-tune acceleration: what your job's kernels actually did

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.54.0/cookbook/notebooks/book/precision/finetune-acceleration.ipynb)

Built from [`cookbook/book/chapters/precision/finetune-acceleration.qmd`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/book/chapters/precision/finetune-acceleration.qmd). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.54.0 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures, from the release's
# tag on GitHub. The chapter runs
# at `small` scale, over the committed samples with a compact text encoder, in
# minutes; the models it runs are downloaded from the Hugging Face Hub on first
# use. SCALE = "full" runs it over the published data and a larger text encoder
# instead, on the GPU: the chapters that fine-tune take hours there.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.54.0", engine + "==0.54.0", "jammi-cookbook @ git+https://github.com/f-inverse/jammi-ai@py-v0.54.0#subdirectory=cookbook/book"], check=True)
SCALE = "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

**Verbs:** `fine_tune(..., backbone_dtype=...)` · `TrainingJob.acceleration_report()` ·
`TrainingJob.metrics()` · **Background:** of the two reduced-precision floats, `bf16`
keeps `f32`'s exponent range and trains without loss scaling (Kalamkar et al. 2019),
where `f16` is the type mixed-precision training first needed loss scaling for
(Micikevicius et al. 2018); FlashAttention computes a whole attention block in
one IO-aware kernel (Dao et al. 2022) · **Checked:** live (a LoRA fine-tune of
ModernBERT on this machine; the job's acceleration report and its kernel dispatch
counters read back and checked against each other).

## A job learns its acceleration when it runs

The training step is built from fused kernels: one GPU or CPU kernel where an eager
implementation would launch several operators — layer norm, the gated feed-forward,
the attention block, the LoRA update, the optimizer step. Whether each one runs
fused depends on the device, the dtype and the model's shapes. A fine-tune is a job,
submitted now and claimed later by whichever worker has capacity, so none of that
is known when you submit. `acceleration_report()` records what was decided, per
job, when a worker claims it: `pending` until then, `determined` after.

The model is ModernBERT (`Alibaba-NLP/gte-modernbert-base`, 149M parameters), with
LoRA adapters on its attention and feed-forward projections, trained on 30 scored
text pairs for one epoch.

In [ ]:
import tempfile

import jammi
from jammi_cookbook import fixtures
from jammi_cookbook.claims import claim

BASE = "Alibaba-NLP/gte-modernbert-base"

db = jammi.connect(f"file://{tempfile.mkdtemp()}")
db.add_source("pairs", url=fixtures.url("tiny_pairs.csv"), format="csv")


def fine_tune(dtype: str):
    return db.fine_tune(source="pairs", base_model=BASE, columns=["text_a", "text_b", "score"],
                        method="lora", task="text_embedding", epochs=1, batch_size=8, seed=0,
                        backbone_dtype=dtype, target_modules=["Wqkv", "Wo", "Wi"])

## Ask for `bf16`

`backbone_dtype` is the dtype the frozen backbone trains in. `bf16` halves its
memory and runs on tensor cores, on a GPU that has them (compute capability 8.0,
Ampere, or newer). On any other device the job is refused, with the reason and the
alternative named, rather than run in another dtype behind your back. So a program
can ask for `bf16` and fall back on the refusal:

In [ ]:
job = fine_tune("bf16")
print(f"just submitted: {job.acceleration_report()}")
try:
    job.wait()
except jammi.errors.TrainingError as refusal:
    print(f"refused: {refusal}")
    claim("bf16 is refused with the reason and the alternatives named",
          all(t in str(refusal) for t in ("bf16", "requires a CUDA device", "f32")))
    job = fine_tune("f32")
    job.wait()
print(f"trained in {job.acceleration_report()['dtype']}")

## Read the report

The report names the device the worker resolved, the dtype, whether this build has
CUDA and FlashAttention compiled in, and, per fused operation, whether it held or
the domain check that turned it down:

In [ ]:
report = job.acceleration_report()
print(f"device {report['device']}   dtype {report['dtype']}   "
      f"cuda compiled {report['cuda_compiled']}   flash compiled {report['flash_compiled']}")
for op, verdict in sorted(report["ops"].items()):
    print(f"  {op:<26} {'held' if verdict['holds'] else 'declined':<9} {verdict['reason']}")
print(f"  {'flash attention':<26} {'held' if report['flash']['holds'] else 'declined':<9} "
      f"{report['flash']['reason']}")

claim("the report is determined once the job has run", report["state"] == "determined")
claim("every declined operation names the check that declined it",
      all(v["holds"] or v["reason"] != "domain_ok" for v in report["ops"].values()))

On a CPU every fused operation holds at `f32`, ModernBERT's attention included —
its 64-wide heads are the shape the fused attention block implements — and
FlashAttention is declined because it is CUDA code: `cuda_not_compiled` on the CPU
build, or a non-CUDA device on a CUDA build. On an Ampere-or-newer GPU with the
CUDA build, `bf16` trains, and FlashAttention takes over the attention block.

In [ ]:
if report["device"].startswith("cuda") and report["flash_compiled"] and report["dtype"] == "bf16":
    claim("on a bf16-capable GPU, FlashAttention runs the attention",
          report["flash"]["holds"], report["flash"]["reason"])
else:
    claim("off a bf16-capable GPU, FlashAttention is declined with its reason",
          not report["flash"]["holds"] and report["flash"]["reason"] != "domain_ok",
          report["flash"]["reason"])

## The counters agree with the report

The report is the decision; the job's metrics count what the kernels then did —
for every fused operation, how many calls ran fused, ran eager, or were declined.
An operation the report says held never fell back to its eager composition, and
when FlashAttention held, its counter shows it ran the attention:

In [ ]:
dispatches = job.metrics()["kernel_dispatches"]
for key, count in sorted(dispatches.items()):
    print(f"  {key:<26} fused {count['fused']:>5}   eager {count['eager']:>5}   "
          f"declined {count['declined']:>5}")

held = [op for op, v in report["ops"].items() if v["holds"] and f"{op}_fused" in dispatches]
claim("no held operation fell back to eager",
      all(dispatches[f"{op}_fused"]["eager"] == 0 for op in held), ", ".join(held))
claim("the held operations ran fused",
      sum(dispatches[f"{op}_fused"]["fused"] for op in held) > 0)
if report["flash"]["holds"]:
    claim("FlashAttention ran the attention", dispatches["attention_block_flash"]["fused"] > 0)

The report is what to read when a fine-tune is slower than it should be: a
declined operation names the check that declined it — the device, the dtype, or a
shape outside the kernel's domain — and that is what to change.

In [ ]:
db.close()

## References

- Kalamkar, Dhiraj, Mudigere, Dheevatsa, Mellempudi, Naveen, Das, Dipankar, Banerjee, Kunal, Avancha, Sasikanth, Vooturi, Dharma Teja, Jammalamadaka, Nataraj, Huang, Jianyu, Yuen, Hector, Yang, Jiyan, Park, Jongsoo, Heinecke, Alexander, Georganas, Evangelos, Srinivasan, Sudarshan, Kundu, Abhisek, Smelyanskiy, Misha, Kaul, Bharat, Dubey, Pradeep (2019) *A Study of BFLOAT16 for Deep Learning Training* arXiv preprint arXiv:1905.12322.
- Micikevicius, Paulius, Narang, Sharan, Alben, Jonah, Diamos, Gregory, Elsen, Erich, Garcia, David, Ginsburg, Boris, Houston, Michael, Kuchaiev, Oleksii, Venkatesh, Ganesh, Wu, Hao (2018) *Mixed Precision Training* International Conference on Learning Representations (ICLR) arXiv:1710.03740.
- Dao, Tri, Fu, Daniel Y., Ermon, Stefano, Rudra, Atri, Ré, Christopher (2022) *FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness* Advances in Neural Information Processing Systems (NeurIPS) arXiv:2205.14135.